# Bankruptcy Prediction with Tuned Neural Networks

Predict which Taiwanese companies go bankrupt from 95 financial ratios. Only about 3% of firms fail, so the question is not accuracy but how many failures a model catches for a given number of false alarms.

The notebook tunes a Keras multilayer perceptron with Keras Tuner and compares it with class-weighted logistic regression and gradient boosting. Every choice (hyperparameters, early stopping, decision threshold) is made on a validation set; the test set is used once at the end. Because positives are rare, the main metric is the area under the precision-recall curve (PR AUC), reported next to ROC AUC.

**Data:** [Taiwanese Bankruptcy Prediction](https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction), UCI Machine Learning Repository, CC BY 4.0: Taiwan Economic Journal, 1999-2009. Liang, D., Lu, C.-C., Tsai, C.-F. and Shih, G.-A. (2016), *European Journal of Operational Research* 252(2).

In [ ]:
import os
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import keras
import keras_tuner as kt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn import metrics
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.utils.class_weight import compute_class_weight

SEED = 2022
keras.utils.set_random_seed(SEED)
sns.set_theme(style="whitegrid")

## Data

In [ ]:
URL = "https://archive.ics.uci.edu/static/public/572/taiwanese+bankruptcy+prediction.zip"
path = Path("data/data.csv")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urlretrieve(URL, "data/bankruptcy.zip")
    zipfile.ZipFile("data/bankruptcy.zip").extractall("data")

ds = pd.read_csv(path)
ds.columns = ds.columns.str.strip()
y = ds.pop("Bankrupt?").to_numpy()
constant = [c for c in ds.columns if ds[c].nunique() == 1]
X = ds.drop(columns=constant)
print(X.shape, "| dropped constant columns:", constant)
print(f"bankrupt: {y.sum()} of {len(y)} ({y.mean():.1%})")

In [ ]:
corr = X.corrwith(pd.Series(y, index=X.index)).abs().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
corr.head(12).sort_values().plot.barh(ax=axes[0], title="Strongest absolute correlations with bankruptcy")
top = corr.index[0]
sns.kdeplot(x=X[top], hue=y, common_norm=False, ax=axes[1]).set_title(top)
plt.tight_layout()
plt.show()

## Split and scale

Stratified 60 / 20 / 20 split. Many ratios are heavy-tailed with outliers several orders of magnitude out, so features are mapped to a normal distribution by quantiles, fitted on the training set only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=SEED)

scaler = QuantileTransformer(output_distribution="normal", n_quantiles=500, random_state=SEED).fit(X_train)
X_train_s, X_val_s, X_test_s = (scaler.transform(a).astype("float32") for a in (X_train, X_val, X_test))
weights = dict(enumerate(compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)))
print({k: v.shape for k, v in {"train": X_train_s, "validation": X_val_s, "test": X_test_s}.items()},
      "| positives:", y_train.sum(), y_val.sum(), y_test.sum(), "| class weights:", {k: round(v, 2) for k, v in weights.items()})

## Baselines

In [ ]:
baselines = {
    "Logistic regression": LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1),
    "Gradient boosting": HistGradientBoostingClassifier(class_weight="balanced", max_iter=300, learning_rate=0.05,
                                                        early_stopping=True, random_state=SEED),
}
scores_val, scores_test = {}, {}
for name, model in baselines.items():
    model.fit(X_train_s, y_train)
    scores_val[name] = model.predict_proba(X_val_s)[:, 1]
    scores_test[name] = model.predict_proba(X_test_s)[:, 1]
    print(f"{name:20s} validation PR AUC {metrics.average_precision_score(y_val, scores_val[name]):.3f}")

## Tuned neural network

Keras Tuner searches depth, width, dropout, L2 penalty and learning rate, maximising validation PR AUC. Each trial trains with class weights and early stopping on the same validation metric.

In [ ]:
def build(hp):
    model = keras.Sequential([keras.Input(shape=(X_train_s.shape[1],))])
    for i in range(hp.Int("layers", 1, 3)):
        model.add(keras.layers.Dense(hp.Choice(f"units_{i}", [16, 32, 64, 128]), activation="relu",
                                     kernel_regularizer=keras.regularizers.l2(hp.Choice("l2", [0.0, 1e-4, 1e-3]))))
        model.add(keras.layers.Dropout(hp.Float("dropout", 0.0, 0.5, step=0.1)))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice("lr", [1e-3, 3e-4, 1e-4])),
                  loss="binary_crossentropy",
                  metrics=[keras.metrics.AUC(curve="PR", name="pr_auc"), keras.metrics.AUC(name="roc_auc")])
    return model

stop = keras.callbacks.EarlyStopping(monitor="val_pr_auc", mode="max", patience=10, restore_best_weights=True)
tuner = kt.RandomSearch(build, objective=kt.Objective("val_pr_auc", "max"), max_trials=20,
                        seed=SEED, directory="tuning", project_name="bankruptcy", overwrite=True)
tuner.search(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=100, batch_size=64,
             class_weight=weights, callbacks=[stop], verbose=0)
best_hp = tuner.get_best_hyperparameters(1)[0]
print({k: v for k, v in best_hp.values.items()})

In [ ]:
mlp = build(best_hp)
history = mlp.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=200, batch_size=64,
                  class_weight=weights, callbacks=[stop], verbose=0)
scores_val["Tuned MLP"] = mlp.predict(X_val_s, verbose=0).ravel()
scores_test["Tuned MLP"] = mlp.predict(X_test_s, verbose=0).ravel()

pd.DataFrame(history.history)[["pr_auc", "val_pr_auc"]].plot(figsize=(8, 4), title="PR AUC by epoch")
plt.xlabel("epoch")
plt.show()

## Test-set comparison

The decision threshold of each model is the one that maximises F1 on the validation set; it is then applied unchanged to the test set.

In [ ]:
def f1_threshold(y_true, score):
    precision, recall, thresholds = metrics.precision_recall_curve(y_true, score)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    return thresholds[np.argmax(f1[:-1])]

rows = {}
for name in scores_test:
    t = f1_threshold(y_val, scores_val[name])
    pred = (scores_test[name] >= t).astype(int)
    tn, fp, fn, tp = metrics.confusion_matrix(y_test, pred).ravel()
    rows[name] = {"PR AUC": metrics.average_precision_score(y_test, scores_test[name]),
                  "ROC AUC": metrics.roc_auc_score(y_test, scores_test[name]),
                  "Threshold": t, "Recall": tp / (tp + fn), "Precision": tp / max(tp + fp, 1),
                  "F1": metrics.f1_score(y_test, pred), "Caught": f"{tp} of {tp + fn}", "False alarms": fp}
results = pd.DataFrame(rows).T.sort_values("PR AUC", ascending=False)
print(f"No-skill PR AUC (positive rate): {y_test.mean():.3f}")
results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for name, s in scores_test.items():
    metrics.PrecisionRecallDisplay.from_predictions(y_test, s, name=name, ax=axes[0])
    metrics.RocCurveDisplay.from_predictions(y_test, s, name=name, ax=axes[1])
axes[0].axhline(y_test.mean(), color="grey", ls="--", label="No skill")
axes[0].set_title("Precision-recall, test set")
axes[1].set_title("ROC, test set")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Sanity checks: no overlap between splits, every model beats the no-skill PR AUC.
assert not set(X_train.index) & set(X_val.index) and not set(X_train.index) & set(X_test.index)
assert (results["PR AUC"].astype(float) > 2 * y_test.mean()).all()

## Conclusion

CONCLUSION_PLACEHOLDER